In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. 전처리된 데이터 불러오기
df = pd.read_csv('../data/polar_weather_preprocessed.csv', parse_dates=['Date'])

# 2. 분석 타겟 설정: 대구 여름 폭염 (장보고기지 12일 래그 적용)
# 장보고기지 기온 데이터를 12일 '미래'로 밀어서 대구 데이터와 동일선상에 맞춤
df['Jangbogo_Temp_Lag12'] = df['Jangbogo_Temp_Mean'].shift(12)

# 3. 정답지(Label) 생성: 대구 불쾌지수 상위 10% 날짜를 '폭염 특보(1)', 나머지는 '정상(0)'으로 정의
threshold_thi = df['Daegu_THI_Max'].quantile(0.90)
df['Is_Heatwave'] = (df['Daegu_THI_Max'] >= threshold_thi).astype(int)

# 4. 결측치 제거 (Shift 하면서 앞부분에 생긴 빈칸 제거)
ml_data = df[['Jangbogo_Temp_Lag12', 'Is_Heatwave']].dropna()

# 5. 문제(X)와 정답(y) 분리
X = ml_data[['Jangbogo_Temp_Lag12']]
y = ml_data['Is_Heatwave']

# 6. 학습용(Train)과 테스트용(Test) 데이터 8:2 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. 랜덤 포레스트(Random Forest) 모델 학습
# n_estimators=100 (100개의 의사결정 나무를 만들어 투표로 결정)
# class_weight='balanced': 적은 데이터(폭염)에 더 높은 가중치를 부여해서 집중 학습시킴
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# 8. 예측 및 성능 평가
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=== 🌡️ 장보고기지 기온 기반 대구 폭염 예측 모델 성능 (가중치 조정 후)===")
print(f"✅ 모델 정확도(Accuracy): {accuracy * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test, y_pred))

=== 🌡️ 장보고기지 기온 기반 대구 폭염 예측 모델 성능 (가중치 조정 후)===
✅ 모델 정확도(Accuracy): 79.70%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       864
           1       0.16      0.30      0.21        82

    accuracy                           0.80       946
   macro avg       0.54      0.57      0.55       946
weighted avg       0.86      0.80      0.82       946



In [5]:
# 1. 추가 파생 변수(Feature Engineering) 생성
# - 장보고기지의 12일 전 '3일 평균 흐름' (단순 1일 데이터의 노이즈를 제거하고 추세를 봄)
df['Jangbogo_Temp_Lag12_MA3'] = df['Jangbogo_Temp_Mean'].shift(12).rolling(window=3).mean()

# - 세종기지 데이터도 12일 래그로 추가 (남극 기류 전체의 복합적인 변화 파악)
df['Sejong_Temp_Lag12'] = df['Sejong_Temp_Mean'].shift(12)

# - 계절성(Seasonality) 반영: 폭염은 주로 여름에 오니까 '월(Month)' 정보가 엄청난 힌트가 됨
df['Month'] = df['Date'].dt.month

# 2. 분석에 사용할 변수들 묶기
features = ['Jangbogo_Temp_Lag12', 'Jangbogo_Temp_Lag12_MA3', 'Sejong_Temp_Lag12', 'Month']

# 결측치 제거 (Shift와 Rolling 때문에 앞부분에 생긴 빈칸 제거)
ml_data_v2 = df[features + ['Is_Heatwave']].dropna()

X_v2 = ml_data_v2[features]
y_v2 = ml_data_v2['Is_Heatwave']

# 3. 데이터 분할 및 모델 재학습
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

# 변수가 늘어났으니 의사결정 나무 개수(n_estimators)를 200개로 넉넉하게 늘려줌
model_v2 = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model_v2.fit(X_train_v2, y_train_v2)

# 4. 성능 평가
y_pred_v2 = model_v2.predict(X_test_v2)
accuracy_v2 = accuracy_score(y_test_v2, y_pred_v2)

print("=== 🚀 대구 지역: 파생 변수 추가 후 예측 모델 성능 (버전 2) ===")
print(f"✅ 모델 정확도(Accuracy): {accuracy_v2 * 100:.2f}%\n")
print("📊 상세 분류 리포트:")
print(classification_report(y_test_v2, y_pred_v2))

# 5. 어떤 변수가 가장 중요했을까? (Feature Importance)
print("-" * 40)
print("🔍 변수 중요도 (어떤 데이터가 정답을 맞히는 데 가장 큰 역할을 했나?)")
importances = model_v2.feature_importances_
for name, imp in zip(features, importances):
    print(f"🔹 {name}: {imp*100:.1f}%")

=== 🚀 대구 지역: 파생 변수 추가 후 예측 모델 성능 (버전 2) ===
✅ 모델 정확도(Accuracy): 93.76%

📊 상세 분류 리포트:
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       864
           1       0.66      0.56      0.60        81

    accuracy                           0.94       945
   macro avg       0.81      0.76      0.79       945
weighted avg       0.93      0.94      0.94       945

----------------------------------------
🔍 변수 중요도 (어떤 데이터가 정답을 맞히는 데 가장 큰 역할을 했나?)
🔹 Jangbogo_Temp_Lag12: 14.7%
🔹 Jangbogo_Temp_Lag12_MA3: 16.3%
🔹 Sejong_Temp_Lag12: 18.5%
🔹 Month: 50.5%
